In [5]:
import duckdb
import pandas as pd
import numpy as np

# connect to GOLD database
con = duckdb.connect(
r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb"
)

# build clean 311 features
con.execute("""
DROP TABLE IF EXISTS features_311_2025_by_acct_clean;

CREATE TABLE features_311_2025_by_acct_clean AS
WITH base AS (
    SELECT
        acct,
        case_type
    FROM warehouse.property_match_311_2025
)
SELECT
    acct,

    COUNT(*) AS cnt_311_total_all,

    SUM(CASE WHEN case_type IN (
        'Weeds/Trash/Stagnant Water on Property',
        'Heavy Trash Code Violation',
        'Trash Dumping or Illegal Dumpsite',
        'Dumpster Complaint',
        'Graffiti on Private/Commercial Property'
    ) THEN 1 ELSE 0 END) AS cnt_311_property_maintenance,

    SUM(CASE WHEN case_type IN (
        'Junk Motor Vehicle',
        'Junk Motor Vehicle - Private Property'
    ) THEN 1 ELSE 0 END) AS cnt_311_vehicle_issues,

    SUM(CASE WHEN case_type IN (
        'Building Code Violation',
        'Dangerous Buildings',
        'Dangerous Commercial Building',
        'Minimum Standards',
        'Minimum Standards - Residence',
        'Occupancy Violation',
        'MultiFamily Habitability Violation',
        'Unregulated Residential Facility',
        'Fire Code Complaint',
        'Health Code'
    ) THEN 1 ELSE 0 END) AS cnt_311_building_standards,

    SUM(CASE WHEN case_type IN (
        'Nuisance On Property',
        'Nuisance on Commercial Property'
    ) THEN 1 ELSE 0 END) AS cnt_311_nuisance

FROM base
GROUP BY acct;
""")

# add total distress score
con.execute("""
DROP TABLE IF EXISTS features_311_2025_by_acct_clean_v2;

CREATE TABLE features_311_2025_by_acct_clean_v2 AS
SELECT
    *,
    (
        cnt_311_property_maintenance +
        cnt_311_vehicle_issues +
        cnt_311_building_standards +
        cnt_311_nuisance
    ) AS cnt_311_total_distress
FROM features_311_2025_by_acct_clean;
""")

# replace table
con.execute("DROP TABLE features_311_2025_by_acct_clean")
con.execute("ALTER TABLE features_311_2025_by_acct_clean_v2 RENAME TO features_311_2025_by_acct_clean")

# update gold 311 feature table
con.execute("""
CREATE OR REPLACE TABLE features_311_by_acct AS
SELECT * FROM features_311_2025_by_acct_clean
""")

# verify
print(con.execute("SHOW TABLES").df())
print(con.execute("SELECT COUNT(*) FROM features_311_by_acct").df())
print(con.execute("SELECT * FROM features_311_by_acct LIMIT 5").df())

                              name
0                       appraisals
1  features_311_2025_by_acct_clean
2             features_311_by_acct
3             features_tax_by_acct
4          ml_tax_delinquency_2025
5                  property_anchor
6       tax_delinquency_2026_clean
7           tax_delinquency_events
   count_star()
0        124339
            acct  cnt_311_total_all  cnt_311_property_maintenance  \
0  0861880000001                  5                           0.0   
1  0861950000006                  4                           0.0   
2  0870020000248                  1                           0.0   
3  0870050000338                  1                           0.0   
4  0870070000373                  7                           0.0   

   cnt_311_vehicle_issues  cnt_311_building_standards  cnt_311_nuisance  \
0                     0.0                         2.0               1.0   
1                     0.0                         0.0               0.0   
2            

In [6]:
import duckdb
import pandas as pd
import numpy as np

# connect to GOLD database
con = duckdb.connect(
    r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb"
)

# rebuild ML dataset using the CLEAN 311 features
ml_df = con.execute("""
SELECT
    a.acct,
    p.appraisal_year AS tax_year,

    a.site_addr_1,
    a.situs_full,
    a.addr_key,
    a.addr_key_no_unit,

    a.owner_entity_flag,
    a.absentee_owner_flag,

    a.Neighborhood_Code AS neighborhood_code,
    a.yr_impr AS year_built,
    a.bld_ar AS building_area,
    a.land_ar AS lot_size,
    a.acreage,

    p.tot_mkt_val AS total_market_value,
    p.protested,
    p.new_own_dt,

    (p.appraisal_year - EXTRACT(YEAR FROM p.new_own_dt)) AS years_owned,

    f.cnt_311_total_all,
    f.cnt_311_property_maintenance,
    f.cnt_311_vehicle_issues,
    f.cnt_311_building_standards,
    f.cnt_311_nuisance,
    f.cnt_311_total_distress,

    t.delinquent_years AS delinquent_years_lifetime,
    t.first_delinquent_year AS first_delinquent_year_lifetime,
    t.most_recent_delinquent_year AS most_recent_delinquent_year_lifetime,
    t.years_since_last_delinquency AS years_since_last_delinquency_lifetime,
    t.delinquent_jurisdiction_records,
    t.delinquency_span_years,
    t.currently_delinquent,
    t.newly_delinquent,
    t.chronic_delinquency

FROM property_anchor a
LEFT JOIN appraisals p
    ON a.acct = p.acct
   AND p.appraisal_year = 2025
LEFT JOIN features_311_by_acct f
    ON a.acct = f.acct
LEFT JOIN features_tax_by_acct t
    ON a.acct = t.acct
""").df()

# -----------------------------
# cleanup / typing
# -----------------------------
ml_df["acct"] = ml_df["acct"].astype(str).str.strip()
ml_df["tax_year"] = pd.to_numeric(ml_df["tax_year"], errors="coerce")

# boolean-like fields
for c in [
    "owner_entity_flag",
    "absentee_owner_flag",
    "currently_delinquent",
    "newly_delinquent",
    "chronic_delinquency",
]:
    ml_df[c] = ml_df[c].fillna(False).astype(int)

# numeric fields
for c in [
    "neighborhood_code",
    "year_built",
    "building_area",
    "lot_size",
    "acreage",
    "total_market_value",
    "years_owned",
    "cnt_311_total_all",
    "cnt_311_property_maintenance",
    "cnt_311_vehicle_issues",
    "cnt_311_building_standards",
    "cnt_311_nuisance",
    "cnt_311_total_distress",
    "delinquent_years_lifetime",
    "first_delinquent_year_lifetime",
    "most_recent_delinquent_year_lifetime",
    "years_since_last_delinquency_lifetime",
    "delinquent_jurisdiction_records",
    "delinquency_span_years",
]:
    ml_df[c] = pd.to_numeric(ml_df[c], errors="coerce")

# fill count-style nulls with zero
for c in [
    "cnt_311_total_all",
    "cnt_311_property_maintenance",
    "cnt_311_vehicle_issues",
    "cnt_311_building_standards",
    "cnt_311_nuisance",
    "cnt_311_total_distress",
    "delinquent_years_lifetime",
    "years_since_last_delinquency_lifetime",
    "delinquent_jurisdiction_records",
    "delinquency_span_years",
]:
    ml_df[c] = ml_df[c].fillna(0)

# derived features
ml_df["property_age"] = ml_df["tax_year"] - ml_df["year_built"]
ml_df.loc[ml_df["property_age"] < 0, "property_age"] = np.nan

ml_df.loc[ml_df["years_owned"] < 0, "years_owned"] = np.nan

ml_df["protested_flag"] = (
    ml_df["protested"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({
        "Y": 1, "YES": 1, "TRUE": 1, "1": 1,
        "N": 0, "NO": 0, "FALSE": 0, "0": 0
    })
    .fillna(0)
    .astype(int)
)

ml_df["value_per_sqft"] = ml_df["total_market_value"] / ml_df["building_area"]
ml_df.loc[~np.isfinite(ml_df["value_per_sqft"]), "value_per_sqft"] = np.nan

# target
ml_df["target_tax_delinquent"] = (
    ml_df["currently_delinquent"]
    .fillna(False)
    .astype(int)
)

# final columns
final_cols = [
    "acct",
    "tax_year",
    "site_addr_1",
    "situs_full",
    "addr_key",
    "addr_key_no_unit",

    "owner_entity_flag",
    "absentee_owner_flag",

    "neighborhood_code",
    "year_built",
    "property_age",
    "years_owned",

    "building_area",
    "lot_size",
    "acreage",

    "total_market_value",
    "value_per_sqft",
    "protested_flag",

    "cnt_311_total_all",
    "cnt_311_property_maintenance",
    "cnt_311_vehicle_issues",
    "cnt_311_building_standards",
    "cnt_311_nuisance",
    "cnt_311_total_distress",

    "delinquent_years_lifetime",
    "first_delinquent_year_lifetime",
    "most_recent_delinquent_year_lifetime",
    "years_since_last_delinquency_lifetime",
    "delinquent_jurisdiction_records",
    "delinquency_span_years",
    "newly_delinquent",
    "chronic_delinquency",

    "target_tax_delinquent",
]

ml_df = ml_df[final_cols].copy()

# save back to gold
con.register("ml_df_view", ml_df)

con.execute("""
CREATE OR REPLACE TABLE ml_tax_delinquency_2025 AS
SELECT * FROM ml_df_view
""")

# verify
print(ml_df.shape)
print(ml_df.head())
print(ml_df["target_tax_delinquent"].value_counts(dropna=False))
print(con.execute("SELECT COUNT(*) FROM ml_tax_delinquency_2025").df())

(1048575, 33)
            acct  tax_year      site_addr_1                     situs_full  \
0  0631170010002      2025  3007 WICHITA ST  3007 WICHITA ST HOUSTON 77004   
1  0631170010003      2025  3009 WICHITA ST  3009 WICHITA ST HOUSTON 77004   
2  0631170010004      2025  3015 WICHITA ST  3015 WICHITA ST HOUSTON 77004   
3  0631170010005      2025  3019 WICHITA ST  3019 WICHITA ST HOUSTON 77004   
4  0631170010010      2025  3115 WICHITA ST  3115 WICHITA ST HOUSTON 77004   

          addr_key addr_key_no_unit  owner_entity_flag  absentee_owner_flag  \
0  3007 WICHITA ST  3007 WICHITA ST                  0                    1   
1  3009 WICHITA ST  3009 WICHITA ST                  0                    0   
2  3015 WICHITA ST  3015 WICHITA ST                  0                    0   
3  3019 WICHITA ST  3019 WICHITA ST                  1                    1   
4  3115 WICHITA ST  3115 WICHITA ST                  0                    1   

   neighborhood_code  year_built  ...  cnt